In [1]:
import pandas as pd

In [2]:
tokyo = pd.read_csv("../data/raw/tokyo_2020_final.csv")

In [3]:
tokyo.columns
tokyo["NOC"].unique()[:10]

array(['Republic of Korea', 'Netherlands', 'Mexico', 'Ecuador', 'Belgium',
       'Slovenia', 'Hungary', 'Italy', "People's Republic of China",
       'Romania'], dtype=object)

In [4]:
codes = pd.read_csv("../data/raw/country_codes.csv")
codes.head()

,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN


In [5]:
codes.columns

Index(['NOC', 'region', 'notes'], dtype='object')

In [6]:
region_to_noc = (
    codes
    .dropna(subset=["region"])
    .set_index("region")["NOC"]
    .to_dict()
)

In [7]:
region_to_noc.get("Japan"), region_to_noc.get("Republic of Korea"), region_to_noc.get("People's Republic of China")

('JPN', None, None)

In [8]:
codes[codes["region"].str.contains("Korea", case=False, na=False)]
codes[codes["region"].str.contains("China", case=False, na=False)]

,NOC,region,notes
41,CHN,China,NaN
88,HKG,China,Hong Kong


In [9]:
name_fix = {
    "Republic of Korea": "South Korea",
    "People's Republic of China": "China",
    "Chinese Taipei": "Chinese Taipei"  # already matches codes as TPE
}

In [10]:
tokyo["NOC"] = tokyo["NOC"].replace(name_fix)

In [11]:
tokyo["NOC_clean"] = tokyo["NOC"].map(region_to_noc)

In [12]:
tokyo[tokyo["NOC_clean"].isna()]["NOC"].value_counts().head(10)

NOC
United States of America    296
ROC                         147
Great Britain               131
Chinese Taipei               16
Hong Kong, China              8
Islamic Republic of Iran      7
Côte d'Ivoire                 1
North Macedonia               1
Syrian Arab Republic          1
Republic of Moldova           1
Name: count, dtype: int64

In [13]:
name_fix_extended = {
    "United States of America": "United States",
    "Great Britain": "United Kingdom",
    "ROC": "Russia",
    "Chinese Taipei": "Chinese Taipei",
    "Hong Kong, China": "Hong Kong",
    "Islamic Republic of Iran": "Iran",
    "Côte d'Ivoire": "Ivory Coast",
    "North Macedonia": "North Macedonia",
    "Syrian Arab Republic": "Syria",
    "Republic of Moldova": "Moldova"
}

In [14]:
tokyo["NOC"] = tokyo["NOC"].replace(name_fix_extended)
tokyo["NOC_clean"] = tokyo["NOC"].map(region_to_noc)

In [15]:
tokyo[tokyo["NOC_clean"].isna()]["NOC"].value_counts()

NOC
United States      296
United Kingdom     131
Chinese Taipei      16
Hong Kong            8
North Macedonia      1
Name: count, dtype: int64

In [16]:
region_to_noc.update({
    "United States": "USA",
    "United Kingdom": "GBR",
    "Chinese Taipei": "TPE",
    "Hong Kong": "HKG",
    "North Macedonia": "MKD"
})

In [17]:
tokyo["NOC_clean"] = tokyo["NOC"].map(region_to_noc)

In [18]:
tokyo[tokyo["NOC_clean"].isna()]["NOC"].value_counts()

Series([], Name: count, dtype: int64)

In [19]:
tokyo.columns


Index(['Name', 'Sex', 'NOC', 'Sport', 'Event', 'Medal', 'Year', 'NOC_clean'], dtype='object')

In [20]:
tokyo["NOC"] = tokyo["NOC_clean"]
tokyo = tokyo.drop(columns=["NOC_clean"])

In [21]:
noc_to_region = (
    codes
    .dropna(subset=["region"])
    .set_index("NOC")["region"]
    .to_dict()
)

tokyo["Country"] = tokyo["NOC"].map(noc_to_region)

In [22]:
tokyo = tokyo[
    ["Name", "Sex", "NOC", "Country", "Sport", "Event", "Medal", "Year"]
]

In [23]:
tokyo.columns
tokyo.head()

,Name,Sex,NOC,Country,Sport,Event,Medal,Year
0,KIM Je Deok,Male,KOR,South Korea,Archery,Mixed Team,Gold Medal,2020
1,AN San,Female,KOR,South Korea,Archery,Mixed Team,Gold Medal,2020
2,SCHLOESSER Gabriela,Female,NED,Netherlands,Archery,Mixed Team,Silver Medal,2020
3,WIJLER Steve,Male,NED,Netherlands,Archery,Mixed Team,Silver Medal,2020
4,ALVAREZ Luis,Male,MEX,Mexico,Archery,Mixed Team,Bronze Medal,2020


In [24]:
tokyo.to_csv("../data/raw/tokyo_2020_final.csv", index=False)